In [1]:
import time
import datetime
import numpy as np
import xarray as xr
from urllib.request import urlopen

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cartopy.crs as ccrs
from shapely.geometry import Point
import matplotlib as mpl
from matplotlib.patches import Rectangle
import geopandas as gpd

In [ ]:
# Satellite PM 2.5 data: 
data5 = gpd.read_file('/data/beronio/WashU_V5_GL/GLv5_GWRPM25_2021-2023_grid_cells.gpkg')

In [2]:
# Translating to Albers Equal Area for the US (EPSG:5070) for accurate distance calculations
data5_trans = data5.to_crs(epsg=5070)

NameError: name 'data5' is not defined

In [ ]:
# Loading EPA monitor dataset: 
epa_monitor = pd.read_csv('/data/beronio/epa_monitor/pm25_monitor_sites_merged_2021_2022_2023_WGS84.csv')

# Columns to keep
columns_to_keep = ['AQS_Site_ID', 'Latitude', 'Longitude', 'Measurement_Scale']

# Keep only relevant columns
gdf_epa_clean = epa_monitor[columns_to_keep].copy()

# Rename columns for simplicity
gdf_epa_clean = gdf_epa_clean.rename(columns={'AQS_Site_ID': 'Site ID', 'Latitude': 'Site Latitude', 'Longitude': 'Site Longitude'})

# Create GEOID by removing the last 4 digits from AQS Site ID + ensure 5-digit GEOID 
gdf_epa_clean['GEOID'] = gdf_epa_clean['Site ID'].astype(str).str.replace('-', '').str[:5]

# Inspect result
print(gdf_epa_clean[['Site ID', 'Site Latitude', 'Site Longitude', 'Measurement_Scale', 'GEOID']].head())

In [ ]:
# Converting monitor sites to point data and transforming to math data5 CRS
gdf_epa_clean['geometry'] = gdf_epa_clean.apply(lambda row: Point(row['Site Longitude'], row['Site Latitude']), axis=1)
gdf_epa_clean = gpd.GeoDataFrame(gdf_epa_clean, geometry='geometry', crs="EPSG:4326")  # correctly label as 4326
gdf_epa_clean = gdf_epa_clean.to_crs(epsg=5070)  # then reproject

In [ ]:
# loading county shapefiles: 
counties = gpd.read_file('/data/beronio/shapefiles/cb_2023_us_county_500k.shp')

# Filtering for the continental US: (using 'STUSPS')

states_to_include = [
    'AL','AZ','AR','CA','CO','CT','DE','FL','GA','ID','DC',
    'IL','IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS',
    'MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK',
    'OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA','WV',
    'WI','WY'
]

counties_conus = counties[counties['STUSPS'].isin(states_to_include)]
counties_conus = counties_conus.drop(["COUNTYNS", 'LSAD', 'ALAND', 'AWATER', 'GEOIDFQ'], axis=1)

# Match CRS to satellite data
counties_conus = counties_conus.to_crs(epsg=5070)

In [ ]:
# Joining county and satellite data: 

grid_with_county = gpd.sjoin(
    data5_trans[['GWRPM25', 'geometry']],
    counties_conus[['GEOID', 'geometry']],
    how='inner',
    predicate='within'
)[['GWRPM25', 'GEOID', 'geometry']]


The below code uses the range (max-min) values to find the top 10 percentile grids for each county

In [3]:

# Step 2: compute county stats (range)
county_stats = (
    grid_with_county.groupby('GEOID')['GWRPM25']
    .agg(
        county_mean='mean',
        county_std='std',
        county_min='min',
        county_max='max',
    )
    .reset_index()
)

# compute range (max - min)
county_stats['range'] = county_stats['county_max'] - county_stats['county_min']

# Step 3: exclude low-variability counties (valid counties > p10 of rel difference [max-min])
q10 = county_stats['range'].quantile(0.10)
print(f"Q10 range threshold: {q10:.3f}")
valid_counties = county_stats[county_stats['range'] > q10]
print(f"Counties kept: {len(valid_counties)}/{len(county_stats)} ({len(valid_counties)/len(county_stats)*100:.1f}%)")

# Step 4: keep only valid counties
grid_with_county_valid = grid_with_county.merge(
    valid_counties[['GEOID']],
    on='GEOID',
    how='inner'
)

# Step 5: recompute stats AFTER filtering
county_stats_valid = (
    grid_with_county_valid.groupby('GEOID')['GWRPM25']
    .agg(
        county_mean='mean',
        county_std='std',
        GWRPM25_90th=lambda x: x.quantile(0.90)
    )
    .reset_index()
)

grid_with_county_valid = grid_with_county_valid.merge(county_stats_valid, on='GEOID')

# Step 6: select hotspots (top 10% only)
top10_sd_hotspots = grid_with_county_valid[
    (grid_with_county_valid['GWRPM25'] >= grid_with_county_valid['GWRPM25_90th']) &
    (~grid_with_county_valid['GEOID'].str.startswith(('02', '15')))
].copy()

NameError: name 'grid_with_county' is not defined

We want to find the hottest counties by concentration: 

In [ ]:
# Locating maximum grid values overall: 

naaqs = 9.0

#Still drop the lowest 10$ of counties by range to remove low variablity counties, but then keep only those above the NAAQS threshold:

county_stats_hot = (
    grid_with_county_valid.groupby('GEOID')['GWRPM25']
    .agg(
        county_mean='mean',
        county_std = 'std',
        GWRPM25_Hot=lambda x: x >= naaqs
    )
)

grid_with_county_hot = grid_with_county_valid.merge(county_stats_hot, on = 'GEOID')

concentration_hotspots = grid_with_county_hot[
    (grid_with_county_hot['GWRPM25'] >= grid_with_county_hot['GWRPM25_Hot'])
].copy()


